# Concept Drift Analysis
Tracks how feature importance changes across temporal groups of training data.  
A feature whose importance **declines significantly over time** indicates P(Y|X) is shifting — i.e. that feature's relationship with churn is becoming stale.

In [ ]:
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from lightgbm import LGBMClassifier

from data_loader import DataLoader
from detection.DriftDetectorV1 import DriftDetectorV1
from preprocessing.BasePreprocessor import Preprocessor

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
FLAGGED_COLOR = "#e74c3c"
STABLE_COLOR  = "#2ecc71"
NEUTRAL_COLOR = "#3498db"

## 1. Load & Preprocess

In [ ]:
loader = DataLoader("../public_data/train.csv", "../public_data/test.csv")
loader.load()
combined = loader.combine()
train_df, test_df = loader.split(combined)

preprocessor = Preprocessor()
train_month_idx = train_df["month_idx"].values
X_train, y_train, _ = preprocessor.fit_transform(train_df, test_df)
feature_names = preprocessor.output_feature_names

print(f"X_train shape : {X_train.shape}")
print(f"Features      : {len(feature_names)}")
print(f"Train months  : {np.unique(train_month_idx)}")

## 2. Run Concept Drift Detection
Using `n_splits=5` for finer temporal granularity than the pipeline default of 3.

In [ ]:
N_SPLITS = 5

detector = DriftDetectorV1()
report = detector.detect_concept_drift(
    X_train, y_train, train_month_idx, feature_names,
    n_splits=N_SPLITS,
)

rows = []
for feat, info in report.items():
    rows.append({
        "feature":         feat,
        "slope":           info["slope"],
        "early_imp":       info["early_imp"],
        "late_imp":        info["late_imp"],
        "relative_change": info["relative_change"],
        "abs_change":      info["abs_change"],
        "direction":       info["direction"],
        "severity":        info["severity"],
        "drifted":         info["drifted"],
        **{f"group_{i}": v for i, v in enumerate(info["importances"])},
    })
summary = pd.DataFrame(rows).sort_values("slope")

n_drifted = summary["drifted"].sum()
print(f"Features analysed : {len(summary)}")
print(f"Flagged as drifted: {n_drifted}  "
      f"(↓ {(summary['drifted'] & (summary['direction']=='decreasing')).sum()} decreasing, "
      f"↑ {(summary['drifted'] & (summary['direction']=='increasing')).sum()} increasing)")
print()
print(summary[summary["drifted"]][["feature","slope","early_imp","late_imp","relative_change","direction","severity"]]
      .sort_values("abs_change", ascending=False).to_string(index=False))

In [ ]:
# Diagnostic: show slope & decline for every feature sorted by slope
# Helps tune thresholds — look for a natural gap between drifted and stable
diag = summary[["feature","slope","relative_decline","drifted"]].copy()
diag["relative_decline"] = diag["relative_decline"].map(lambda v: f"{v:.1%}")
diag["slope"] = diag["slope"].map(lambda v: f"{v:.3f}")
print(diag.sort_values("slope").to_string(index=False))

## 3. Churn Rate Over Time  
P(Y) shift — the base rate of churn moving across months.

In [ ]:
churn_by_month = (
    train_df.assign(churned=lambda d: (d["ChurnStatus"].str.strip().str.lower() == "yes").astype(int))
    .groupby("Month")
    .agg(churn_rate=("churned", "mean"), n=("churned", "count"))
    .reset_index()
)
churn_by_month["month_dt"] = pd.to_datetime(churn_by_month["Month"], format="%y-%b")
churn_by_month = churn_by_month.sort_values("month_dt")

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(churn_by_month["Month"], churn_by_month["churn_rate"], marker="o", color=NEUTRAL_COLOR, linewidth=2)
ax.axhline(churn_by_month["churn_rate"].mean(), linestyle="--", color="gray", alpha=0.7, label=f'Mean {churn_by_month["churn_rate"].mean():.1%}')
ax.fill_between(range(len(churn_by_month)), churn_by_month["churn_rate"], alpha=0.1, color=NEUTRAL_COLOR)
ax.set_title("Churn Rate Over Time (P(Y) shift)", fontsize=14, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Churn Rate")
ax.set_xticks(range(len(churn_by_month)))
ax.set_xticklabels(churn_by_month["Month"], rotation=45, ha="right")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.1%}"))
ax.legend()
plt.tight_layout()
plt.show()

## 4. Feature Importance Heatmap  
Each column is a temporal group (early → late). Darker = higher normalised importance. Flagged features are shown in red on the y-axis.

In [ ]:
group_cols = [c for c in summary.columns if c.startswith("group_")]
heatmap_df = summary.set_index("feature")[group_cols].copy()
heatmap_df.columns = [f"Group {i+1}" for i in range(len(group_cols))]

# Sort: flagged first, then by slope within each group
heatmap_df = heatmap_df.loc[
    summary.sort_values(["drifted", "slope"], ascending=[False, True])["feature"]
]

fig, ax = plt.subplots(figsize=(max(8, len(group_cols) * 1.5), max(8, len(heatmap_df) * 0.35)))
sns.heatmap(
    heatmap_df, ax=ax, cmap="YlOrRd", annot=True, fmt=".3f",
    linewidths=0.4, linecolor="white",
    cbar_kws={"label": "Normalised Importance", "shrink": 0.6},
)
ax.set_title("Feature Importance Across Temporal Groups", fontsize=14, fontweight="bold", pad=14)
ax.set_xlabel("Temporal Group (early → late)")
ax.set_ylabel("")

# Colour y-axis labels: red = flagged, green = stable
flagged_set = set(summary[summary["drifted"]]["feature"])
for label in ax.get_yticklabels():
    label.set_color(FLAGGED_COLOR if label.get_text() in flagged_set else "black")
    label.set_fontweight("bold" if label.get_text() in flagged_set else "normal")

plt.tight_layout()
plt.show()

## 5. Importance Trajectory — All Features  
Line per feature showing normalised importance across temporal groups. Red = flagged for concept drift.

In [ ]:
n_features = len(summary)
n_cols = 3
n_rows = (n_features + n_cols - 1) // n_cols
x_ticks = [f"G{i+1}" for i in range(len(group_cols))]
x = np.arange(len(group_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 3))
axes = axes.flatten()

for idx, (_, row) in enumerate(summary.sort_values(["drifted", "slope"], ascending=[False, True]).iterrows()):
    ax = axes[idx]
    imp = [row[g] for g in group_cols]
    color = FLAGGED_COLOR if row["drifted"] else NEUTRAL_COLOR

    ax.plot(x, imp, marker="o", color=color, linewidth=2)
    ax.fill_between(x, imp, alpha=0.12, color=color)

    title = row["feature"]
    if row["drifted"]:
        title += f"  ▼{row['relative_decline']:.0%}"
    ax.set_title(title, fontsize=9, fontweight="bold", color=color)
    ax.set_xticks(x)
    ax.set_xticklabels(x_ticks, fontsize=8)
    ax.set_ylabel("Norm. Importance", fontsize=7)
    ax.tick_params(axis="y", labelsize=7)

    # Trend line
    z = np.polyfit(x, imp, 1)
    ax.plot(x, np.poly1d(z)(x), linestyle="--", color=color, alpha=0.4, linewidth=1)

# Hide unused subplots
for idx in range(n_features, len(axes)):
    axes[idx].set_visible(False)

fig.suptitle("Feature Importance Trajectories (red = concept drift flagged)", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 6. Relative Decline Bar Chart  
How much importance each feature lost from Group 1 → last group, as a percentage of its initial importance. The dashed line is the 30% flagging threshold.

In [ ]:
plot_df = summary.sort_values("relative_decline", ascending=False)
colors = [FLAGGED_COLOR if d else NEUTRAL_COLOR for d in plot_df["drifted"]]

fig, ax = plt.subplots(figsize=(12, max(5, len(plot_df) * 0.38)))
bars = ax.barh(plot_df["feature"], plot_df["relative_decline"], color=colors, edgecolor="white", height=0.7)
ax.axvline(0.30, linestyle="--", color="gray", linewidth=1.2, label="30% threshold")

for bar, val in zip(bars, plot_df["relative_decline"]):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
            f"{val:.0%}", va="center", fontsize=8)

ax.set_xlabel("Relative Importance Decline (Group 1 → Last Group)")
ax.set_title("Relative Concept Drift Decline per Feature", fontsize=13, fontweight="bold")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_xlim(0, plot_df["relative_decline"].max() * 1.2)
ax.invert_yaxis()
ax.legend()

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=FLAGGED_COLOR, label="Flagged (concept drift)"),
                   Patch(facecolor=NEUTRAL_COLOR, label="Stable")]
ax.legend(handles=legend_elements, loc="lower right")

plt.tight_layout()
plt.show()

## 7. Slope vs Relative Decline Scatter  
Each dot is a feature. Top-left quadrant (steep negative slope + high decline) = most concept-drifted.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))

for _, row in summary.iterrows():
    color = FLAGGED_COLOR if row["drifted"] else NEUTRAL_COLOR
    ax.scatter(row["slope"], row["relative_decline"], color=color, s=80, zorder=3, alpha=0.85)
    ax.annotate(
        row["feature"], (row["slope"], row["relative_decline"]),
        fontsize=7.5, textcoords="offset points", xytext=(5, 3), color=color,
    )

# Threshold lines
ax.axvline(-0.02, linestyle="--", color="gray", alpha=0.6, linewidth=1, label="slope threshold (−0.02)")
ax.axhline(0.30,  linestyle="--", color="gray", alpha=0.6, linewidth=1, label="decline threshold (30%)")

# Shade the flagging quadrant
ax.axvspan(summary["slope"].min() - 0.005, -0.02,
           ymin=(0.30 - ax.get_ylim()[0]) / (ax.get_ylim()[1] - ax.get_ylim()[0]) if ax.get_ylim()[1] != ax.get_ylim()[0] else 0,
           alpha=0.06, color=FLAGGED_COLOR)

ax.set_xlabel("Importance Slope (negative = declining over time)", fontsize=11)
ax.set_ylabel("Relative Decline (Group 1 → Last)", fontsize=11)
ax.set_title("Concept Drift: Slope vs Relative Decline", fontsize=13, fontweight="bold")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=FLAGGED_COLOR, label="Flagged"),
    Patch(facecolor=NEUTRAL_COLOR, label="Stable"),
]
ax.legend(handles=legend_elements)
plt.tight_layout()
plt.show()

## 8. Summary Table

In [ ]:
display_cols = ["feature", "slope", "relative_decline", "severity", "drifted"] + group_cols
tbl = summary[display_cols].copy()
tbl["relative_decline"] = tbl["relative_decline"].map(lambda v: f"{v:.1%}")
tbl["slope"] = tbl["slope"].map(lambda v: f"{v:.4f}")
tbl.columns = ["Feature", "Slope", "Relative Decline", "Severity", "Flagged"] + [f"Group {i+1}" for i in range(len(group_cols))]

severity_colors = {"low": "#fff3cd", "moderate": "#ffe0b2", "high": "#ffccbc", "very high": "#fde8e8", "none": ""}

def highlight_row(row):
    color = severity_colors.get(row["Severity"], "")
    return [f"background-color: {color}" if color else "" for _ in row]

(tbl.sort_values("Flagged", ascending=False)
    .style
    .apply(highlight_row, axis=1)
    .format({f"Group {i+1}": "{:.4f}" for i in range(len(group_cols))})
    .set_properties(**{"font-size": "11px"})
)